# BIOT 6900 · Module 2 — Multi-Omics Target Identification & Validation
### Week 2 · Computational Lab 2 · Starter Notebook

**What this notebook does.** You'll integrate three data layers — transcriptomics (RNA), proteomics (protein), and genomics — to nominate and *rank* disease targets, then hand that ranked list to Weeks 3–4.

**How today is built — I do → we do → you do:**
- **Part 1 (worked, already complete):** CPTAC **breast cancer** on *synthetic* data. The tumors are measured at all three layers, so the data is *sample-matched* — you integrate **at the sample level**. Your instructor runs these cells; follow along.
- **Part 2 (together, in class):** find and load *real* CPTAC data, then re-run Part 1 on real numbers. Not graded.
- **Part 3 (your assignment, `# TODO` cells):** **Alzheimer's disease**. The cohorts are *not* matched across layers, so you integrate **at the gene level**. You transfer the Part 1 method to messier, more realistic data.

> **The one idea to carry through:** *how* you integrate is dictated by *what your data has matched.* Matched → correlate within samples. Unmatched → compare gene-level summaries. Same goal, different machinery.

**Data.** Part 1 looks for the synthetic CPTAC files in `data/` and, if they're absent, falls back to clearly-labelled demo data so it always runs. Part 3 requires the three Alzheimer's matrices posted on **Canvas** — place them in `data/` (see the Lab Guide).

*Continuity from Module 1:* you'll see **TP53 / p53** (`P04637`, PDB `1TUP`) surface in the cancer half and **APOE** (`rs7412`) in the Alzheimer's half.

**Before you submit:** `Kernel → Restart & Run All` so the whole notebook executes top to bottom.

In [1]:
import os
import numpy as np
import pandas as pd
from scipy import stats

RNG = np.random.default_rng(6900)  # fixed seed so results are reproducible

# Module 1 continuity anchors
P53_UNIPROT, P53_PDB = "P04637", "1TUP"   # p53  -> cancer half
APOE_VARIANT = "rs7412"                    # APOE -> Alzheimer's half

# Default scoring weights for this course: equal across the three layers.
# You MAY change these in Part 3 — but if you do, you must justify it in your report.
EQUAL_WEIGHTS = {"transcriptomic": 1/3, "proteomic": 1/3, "genomic": 1/3}

## Helper functions you'll use in both parts
Two small functions do the scoring work. `rank_percentile` puts any layer's scores on a common 0–1 scale (by rank, so it's robust to outliers and to layers being on different units). `multi_evidence_score` normalizes each layer and takes the weighted sum.

In [2]:
def rank_percentile(series: pd.Series) -> pd.Series:
    """Normalize any score to [0, 1] by rank. Robust to outliers and scale differences."""
    return series.rank(method="average", pct=True)


def multi_evidence_score(df: pd.DataFrame, cols, weights) -> pd.Series:
    """Weighted sum of rank-percentile-normalized layer scores."""
    normed = pd.DataFrame({c: rank_percentile(df[c]) for c in cols})
    return sum(weights[c] * normed[c] for c in cols)

## Data loading (provided — you don't need to edit these)
`load_cptac()` reads the matched breast-cancer matrices for Part 1 (or synthesizes demo data if they're missing). `load_ad()` reads the three Alzheimer's matrices you download from Canvas for Part 2.

In [3]:
CPTAC_GENES = ["TP53", "PIK3CA", "ERBB2", "ESR1", "GATA3", "MYC", "CDH1",
               "MAP3K1", "PTEN", "AKT1", "RB1", "CCND1", "FOXA1", "MKI67",
               "EGFR", "BRCA1", "BRCA2", "KRT5", "VIM", "ACTB"]


def _synth_cptac(n_tumor=60, n_normal=15):
    genes = CPTAC_GENES
    tumor_ids = [f"T{i:02d}" for i in range(n_tumor)]
    normal_ids = [f"N{i:02d}" for i in range(n_normal)]
    coupling = pd.Series(RNG.uniform(0.05, 0.80, len(genes)), index=genes)

    def gene_matrix(sample_ids, shift):
        rna = pd.DataFrame(RNG.normal(0, 1, (len(genes), len(sample_ids))),
                           index=genes, columns=sample_ids).add(shift, axis=0)
        noise = pd.DataFrame(RNG.normal(0, 1, rna.shape), index=genes, columns=sample_ids)
        prot = rna.mul(coupling, axis=0) + noise.mul(1 - coupling + 0.35, axis=0)
        return rna, prot

    shift = pd.Series(0.0, index=genes)
    for g in ["TP53", "ERBB2", "MKI67", "MYC", "PIK3CA"]:
        shift[g] = RNG.uniform(1.2, 2.2)
    rna_t, prot_t = gene_matrix(tumor_ids, shift)
    rna_n, prot_n = gene_matrix(normal_ids, pd.Series(0.0, index=genes))
    rna = pd.concat([rna_t, rna_n], axis=1)
    prot = pd.concat([prot_t, prot_n], axis=1)
    mut_freq = pd.Series(RNG.uniform(0.0, 0.05, len(genes)), index=genes)
    for g in ["TP53", "PIK3CA", "CDH1", "GATA3"]:
        mut_freq[g] = RNG.uniform(0.25, 0.45)
    meta = pd.Series(["tumor"] * n_tumor + ["normal"] * n_normal,
                     index=tumor_ids + normal_ids, name="group")
    return rna, prot, mut_freq, meta


def load_cptac():
    p = {"rna": "data/cptac_brca_rna.tsv", "prot": "data/cptac_brca_protein.tsv",
         "mut": "data/cptac_brca_mutation.tsv"}
    if all(os.path.exists(v) for v in p.values()):
        rna = pd.read_csv(p["rna"], sep="\t", index_col=0)
        prot = pd.read_csv(p["prot"], sep="\t", index_col=0)
        mut = pd.read_csv(p["mut"], sep="\t", index_col=0).iloc[:, 0]
        print("Loaded CPTAC files from data/.")
        return rna, prot, mut, None
    print("!! WARNING: CPTAC files not found -> SYNTHETIC demo data "
          "(illustrative only, not real CPTAC values).")
    return _synth_cptac()


def load_ad():
    p = {"tx": "data/ad_transcriptomics.tsv", "pr": "data/ad_proteomics.tsv",
         "gw": "data/ad_gwas.tsv"}
    missing = [v for v in p.values() if not os.path.exists(v)]
    if missing:
        raise FileNotFoundError(
            "Alzheimer's matrices not found: " + ", ".join(missing) +
            "\nDownload the three files from Canvas and put them in a 'data/' folder "
            "next to this notebook (see the Lab Guide).")
    return (pd.read_csv(p["tx"], sep="\t"),
            pd.read_csv(p["pr"], sep="\t"),
            pd.read_csv(p["gw"], sep="\t"))

---
# Part 1 — Worked example: CPTAC breast cancer *(run and read; nothing to edit)*

**Why integrate at all?** Each layer can produce artifacts the others don't share, so a target supported across genome, transcriptome, and proteome is a stronger bet than one seen in only one layer. Integration is triangulation.

Because CPTAC measures the **same tumors** at every layer, we can line samples up and integrate **at the sample level.**

In [4]:
rna, prot, mut_freq, meta = load_cptac()
print("RNA matrix:    ", rna.shape, "(genes x samples)")
print("Protein matrix:", prot.shape)
print("Mutation freq: ", mut_freq.shape)

Loaded CPTAC files from data/.
RNA matrix:     (23121, 122) (genes x samples)
Protein matrix: (12621, 122)
Mutation freq:  (9448,)


### 1.2 Harmonize identifiers → a common gene key
The real technical wall in multi-omics: gene symbols, Ensembl IDs, and UniProt accessions don't line up for free. Here we intersect on a shared key and check how many genes *survive the join* — always your first reality check.

In [5]:
common = rna.index.intersection(prot.index).intersection(mut_freq.index)
print(f"Genes surviving the 3-way join: {len(common)} "
      f"(RNA {len(rna.index)}, protein {len(prot.index)}, mutation {len(mut_freq.index)})")
rna, prot, mut_freq = rna.loc[common], prot.loc[common], mut_freq.loc[common]
samples = rna.columns.intersection(prot.columns)

Genes surviving the 3-way join: 6306 (RNA 23121, protein 12621, mutation 9448)


### 1.3–1.4 Sample-level RNA–protein correlation
With matched samples we can ask, per gene, *does protein track RNA across patients?* The answer is usually **partial** — correlation is modest and varies by gene. A gene where protein does **not** track RNA isn't broken data; it's a signal of post-transcriptional regulation (translational buffering, protein turnover).

In [6]:
corr = pd.Series(
    {g: stats.spearmanr(rna.loc[g, samples], prot.loc[g, samples]).statistic
     for g in common}, name="rna_prot_corr")
print(f"RNA-protein correlation: median={corr.median():.2f}, "
      f"range=[{corr.min():.2f}, {corr.max():.2f}]   <- note: NOT ~1.0")
print(f"  most coupled: {corr.idxmax()} ({corr.max():.2f});  "
      f"most buffered: {corr.idxmin()} ({corr.min():.2f})")

RNA-protein correlation: median=0.48, range=[-0.23, 0.92]   <- note: NOT ~1.0
  most coupled: VWA5A (0.92);  most buffered: ARPC1A (-0.23)


**Read this result.** The median correlation is well below 1 — most genes' protein levels only partly follow their RNA. That is the empirical reason you can't treat RNA as a stand-in for protein, and why integrating both layers adds information.

### 1.5 Per-layer differential signal
Each layer independently nominates candidates. Here the signal is the tumor-vs-normal effect size at RNA and protein, plus per-gene mutation frequency for the genomic layer.

In [7]:
if meta is not None:
    tcols = meta.index[meta == "tumor"]
    ncols = meta.index[meta == "normal"]
    rna_eff = (rna[tcols].mean(axis=1) - rna[ncols].mean(axis=1)).abs()
    prot_eff = (prot[tcols].mean(axis=1) - prot[ncols].mean(axis=1)).abs()
else:
    rna_eff = rna[samples].mean(axis=1).abs()
    prot_eff = prot[samples].mean(axis=1).abs()

scored = pd.DataFrame({"transcriptomic": rna_eff, "proteomic": prot_eff,
                       "genomic": mut_freq, "rna_prot_corr": corr})
scored.round(3).head()

,transcriptomic,proteomic,genomic,rna_prot_corr
A2M,0.060,0.494,0.033,0.349
A2ML1,0.954,3.147,0.016,NaN
AADACL2,0.499,0.396,0.008,NaN
AAED1,0.070,0.434,0.016,NaN
AAGAB,0.026,0.211,0.008,0.666


### 1.6 Multi-evidence score → ranked targets
Normalize each layer to a common scale and take the **equal-weighted** sum. This turns three noisy layers into one ranked list — the artifact everything downstream consumes.

In [8]:
scored["score"] = multi_evidence_score(
    scored, ["transcriptomic", "proteomic", "genomic"], EQUAL_WEIGHTS)
ranked_cptac = scored.sort_values("score", ascending=False)
print("Top CPTAC targets (worked example):")
print(ranked_cptac.head(6).round(3).to_string())
# Continuity check: TP53 / p53 (Module 1's P04637 / 1TUP) is a top breast-cancer hit.
print("\nTP53 rank:", list(ranked_cptac.index).index("TP53") + 1)

Top CPTAC targets (worked example):
         transcriptomic  proteomic  genomic  rna_prot_corr  score
SI                0.977      5.200    0.049            NaN  0.988
VWDE              1.263      1.424    0.049            NaN  0.979
MUC5B             0.566      3.441    0.082            NaN  0.978
SPHKAP            0.573      3.321    0.041            NaN  0.970
CEACAM5           0.813      3.315    0.033            NaN  0.970
RIMS2             1.240      1.676    0.033            NaN  0.968

TP53 rank: 108


### Part 1 recap
You integrated **at the sample level** because the data was **matched** — you could correlate RNA and protein within the same tumors. That was synthetic data, so the method is easy to see. Next we run it on real data, then you transfer it to unmatched data yourself.

---
# Part 2 — Find and load real CPTAC data *(together, in class)*

Now swap the synthetic files for the real thing. CPTAC breast data is open access (no data-use agreement) — we'll get it together in class. **Not graded.**

1. Go to **LinkedOmics** (`linkedomics.org`) → **CPTAC Breast Cancer (BRCA)**.
2. Download the gene-level **RNAseq** and **Proteome** matrices (gene × sample).
3. Save them into `data/` with genes as the row index and these exact names — transpose with `.T` first if samples are in rows:
   - `data/cptac_brca_rna.tsv`
   - `data/cptac_brca_protein.tsv`
   - `data/cptac_brca_mutation.tsv`  (one row per gene: mutation frequency or CNV magnitude)
4. `Kernel → Restart & Run All` and watch **Part 1** again — it now loads your real files instead of the demo. Compare: is the RNA–protein correlation still modest? Does TP53 still surface near the top?

*Wrinkle:* the real download has no tumor/normal labels, so ranking falls back to overall abundance rather than a tumor-vs-normal effect. That's fine for seeing the pipeline run on real data.

---
# Part 3 — Your assignment: Ovarian's disease *(complete the `# TODO` cells)*

The three matrices you downloaded from Canvas are **gene-level summaries** from different cohorts — there are no shared samples to correlate, so you integrate **at the gene level.** You'll transfer the Part 1 workflow: harmonize → concordance → score → rank → export.

Your ranked target list (the CSV you export in 3.5) is what **Week 3** builds on, so it has to be clean. *Continuity:* expect **APOE** (`rs7412`) among your top hits.

### 3.1 — TODO: load and inspect the three matrices
Call `load_ad()` and look at each table's columns and shape before you touch them.

In [9]:
# TODO 3.1 — load the three ovarian cancer matrices and inspect them.

import pandas as pd

# RNA differential expression
rna_de = pd.read_csv(
    "raw_ov/E-GEOD-40595_A-AFFY-44-analytics.tsv",
    sep="\t"
)

# Protein tumor
prot_tumor = pd.read_csv(
    "raw_ov/OV_proteomics_gene_abundance_log2_reference_intensity_normalized_Tumor.txt",
    sep="\t",
    index_col=0
)

# Protein normal
prot_normal = pd.read_csv(
    "raw_ov/OV_proteomics_gene_abundance_log2_reference_intensity_normalized_Normal.txt",
    sep="\t",
    index_col=0
)

# Mutation
gw = pd.read_csv(
    "raw_ov/OV_somatic_mutation_gene_level_binary.txt",
    sep="\t",
    index_col=0
)

print("RNA differential:", rna_de.shape)
print("Protein tumor:", prot_tumor.shape)
print("Protein normal:", prot_normal.shape)
print("Mutation:", gw.shape)

display(rna_de.head())

RNA differential: (21056, 9)
Protein tumor: (10589, 83)
Protein normal: (10587, 19)
Mutation: (8298, 82)


,Gene ID,Gene Name,Design Element,g1_g2.p-value,g1_g2.t-statistic,g1_g2.log2foldchange,g3_g4.p-value,g3_g4.t-statistic,g3_g4.log2foldchange
0,ENSG00000000003,TSPAN6,209108_at,0.004465,-3.236672,-1.1,0.004780,3.327700,1.6
1,ENSG00000000005,TNMD,220065_at,0.004992,3.191532,0.3,0.033941,-2.443501,-0.6
2,ENSG00000000419,DPM1,202673_at,0.040081,-2.287546,-0.8,0.102456,1.871482,0.8
3,ENSG00000000457,SCYL3,41329_at,0.013488,-2.777917,-0.7,0.000009,5.869504,1.5
4,ENSG00000000460,C1orf112,220840_s_at,0.011583,2.843111,0.3,0.801533,0.304991,0.0


In [10]:
# 3.2 — prepare RNA gene-level differential expression table

rna = rna_de[
    ["Gene Name", "g1_g2.log2foldchange", "g1_g2.p-value"]
].copy()

rna.columns = ["gene", "rna_log2fc", "rna_pval"]

# Remove rows without a gene symbol
rna = rna.dropna(subset=["gene"])

print("RNA rows:", len(rna))
print("Duplicate gene symbols:", rna["gene"].duplicated().sum())

display(rna.head())

RNA rows: 19128
Duplicate gene symbols: 34


,gene,rna_log2fc,rna_pval
0,TSPAN6,-1.1,0.004465
1,TNMD,0.3,0.004992
2,DPM1,-0.8,0.040081
3,SCYL3,-0.7,0.013488
4,C1orf112,0.3,0.011583


In [11]:
# Collapse duplicate RNA gene symbols to one row per gene

rna = (
    rna.groupby("gene", as_index=False)
       .agg({
           "rna_log2fc": "mean",
           "rna_pval": "min"
       })
)

print("RNA genes after collapsing duplicates:", len(rna))
print("Duplicate gene symbols:", rna["gene"].duplicated().sum())

display(rna.head())

RNA genes after collapsing duplicates: 19094
Duplicate gene symbols: 0


,gene,rna_log2fc,rna_pval
0,A1BG-AS1,-0.1,2.640360e-01
1,A1CF,1.2,7.297924e-08
2,A2M,-1.6,1.674185e-02
3,A2M-AS1,-0.5,1.187931e-02
4,A2ML1,0.8,3.730677e-11


### 3.2 — TODO: harmonize on the gene symbol → one joined table
Rename the effect/`pval` columns so RNA and protein don't collide, then merge all three on `gene`. Report how many genes survive the join (your first reality check).

In [12]:
from scipy import stats
import pandas as pd

# 3.2 — prepare protein gene-level effect and p-value

# Remove Ensembl version suffixes
prot_tumor.index = prot_tumor.index.str.split(".").str[0]
prot_normal.index = prot_normal.index.str.split(".").str[0]

# Keep genes present in both tumor and normal
common_prot_genes = prot_tumor.index.intersection(prot_normal.index)

prot_t = prot_tumor.loc[common_prot_genes]
prot_n = prot_normal.loc[common_prot_genes]

# Effect = mean(tumor) - mean(normal)
prot_effect = prot_t.mean(axis=1) - prot_n.mean(axis=1)

# Welch t-test for each gene
prot_pval = pd.Series({
    gene: stats.ttest_ind(
        prot_t.loc[gene].dropna(),
        prot_n.loc[gene].dropna(),
        equal_var=False
    ).pvalue
    for gene in common_prot_genes
})

protein = pd.DataFrame({
    "ensembl": common_prot_genes,
    "prot_effect": prot_effect.values,
    "prot_pval": prot_pval.values
})

print("Protein genes:", len(protein))
display(protein.head())

Protein genes: 10587


,ensembl,prot_effect,prot_pval
0,ENSG00000000003,0.493981,0.001647
1,ENSG00000000419,0.316248,0.000824
2,ENSG00000000457,0.627962,0.018564
3,ENSG00000000460,0.480944,0.747531
4,ENSG00000000938,-0.657602,0.000894


In [13]:
import mygene

mg = mygene.MyGeneInfo()

results = mg.querymany(
    protein["ensembl"].tolist(),
    scopes="ensembl.gene",
    fields="symbol",
    species="human"
)

id_to_symbol = {
    r["query"]: r["symbol"]
    for r in results
    if "symbol" in r and not r.get("notfound", False)
}

protein["gene"] = protein["ensembl"].map(id_to_symbol)

# Keep successfully mapped genes
protein = protein.dropna(subset=["gene"])

print("Protein genes after symbol mapping:", len(protein))
print("Duplicate gene symbols:", protein["gene"].duplicated().sum())

display(protein.head())

5 input query terms found no hit:	['ENSG00000130723', 'ENSG00000148362', 'ENSG00000215271', 'ENSG00000263264', 'ENSG00000268861']


Protein genes after symbol mapping: 10579
Duplicate gene symbols: 0


,ensembl,prot_effect,prot_pval,gene
0,ENSG00000000003,0.493981,0.001647,TSPAN6
1,ENSG00000000419,0.316248,0.000824,DPM1
2,ENSG00000000457,0.627962,0.018564,SCYL3
3,ENSG00000000460,0.480944,0.747531,FIRRM
4,ENSG00000000938,-0.657602,0.000894,FGR


In [14]:
# Prepare genomics layer: mutation frequency per gene

# Remove Ensembl version suffixes
gw.index = gw.index.str.split(".").str[0]

# Mutation frequency across tumor samples
mutation_freq = gw.mean(axis=1)

genomics = pd.DataFrame({
    "ensembl": mutation_freq.index,
    "mutation_freq": mutation_freq.values
})

print("Genomic genes:", len(genomics))
display(genomics.head())

Genomic genes: 8298


,ensembl,mutation_freq
0,ENSG00000000971,0.012195
1,ENSG00000001036,0.012195
2,ENSG00000001084,0.012195
3,ENSG00000001461,0.012195
4,ENSG00000001561,0.012195


In [15]:
results = mg.querymany(
    genomics["ensembl"].tolist(),
    scopes="ensembl.gene",
    fields="symbol",
    species="human"
)

id_to_symbol_g = {
    r["query"]: r["symbol"]
    for r in results
    if "symbol" in r and not r.get("notfound", False)
}

genomics["gene"] = genomics["ensembl"].map(id_to_symbol_g)

# Keep successfully mapped genes
genomics = genomics.dropna(subset=["gene"])

print("Genomic genes after symbol mapping:", len(genomics))
print("Duplicate gene symbols:", genomics["gene"].duplicated().sum())

display(genomics.head())

2 input query terms found dup hits:	[('ENSG00000117262', 2), ('ENSG00000188092', 2)]
9 input query terms found no hit:	['ENSG00000130723', 'ENSG00000183729', 'ENSG00000184293', 'ENSG00000225932', 'ENSG00000243135', 'ENS


Genomic genes after symbol mapping: 8279
Duplicate gene symbols: 1


,ensembl,mutation_freq,gene
0,ENSG00000000971,0.012195,CFH
1,ENSG00000001036,0.012195,FUCA2
2,ENSG00000001084,0.012195,GCLC
3,ENSG00000001461,0.012195,NIPAL3
4,ENSG00000001561,0.012195,ENPP4


In [16]:
# Collapse duplicate genomic gene symbols

genomics = (
    genomics.groupby("gene", as_index=False)
            .agg({
                "mutation_freq": "mean"
            })
)

print("Genomic genes after collapsing duplicates:", len(genomics))
print("Duplicate gene symbols:", genomics["gene"].duplicated().sum())

display(genomics.head())

Genomic genes after collapsing duplicates: 8278
Duplicate gene symbols: 0


,gene,mutation_freq
0,A1CF,0.012195
1,A2M,0.012195
2,A2ML1,0.012195
3,A4GALT,0.012195
4,A4GNT,0.012195


In [17]:
# Merge RNA, protein, and genomics on gene

df = (
    rna
    .merge(
        protein[["gene", "prot_effect", "prot_pval"]],
        on="gene",
        how="inner"
    )
    .merge(
        genomics[["gene", "mutation_freq"]],
        on="gene",
        how="inner"
    )
)

print("Genes surviving the 3-way join:", len(df))
display(df.head())

Genes surviving the 3-way join: 4396


,gene,rna_log2fc,rna_pval,prot_effect,prot_pval,mutation_freq
0,A2M,-1.6,1.674185e-02,-1.378499,5.591408e-07,0.012195
1,A2ML1,0.8,3.730677e-11,-1.477915,1.910683e-02,0.012195
2,A4GALT,0.2,1.636356e-01,-0.409790,NaN,0.012195
3,AACS,-0.3,1.966716e-01,0.120361,3.812444e-01,0.012195
4,AADAC,-2.9,4.008839e-07,-0.454076,4.798840e-01,0.012195


### 3.3 — TODO: sign-agreement concordance
You can't correlate across samples here (nothing is matched), so concordance becomes **direction agreement**: does the gene move the *same way* at RNA and protein? Add a boolean `concordant` column. Watch for genes that are strong at RNA but flat/opposite at protein — those are the interesting discordant ones.

In [18]:
# 3.3 — sign-agreement concordance

df["concordant"] = (
    (df["rna_log2fc"] > 0) & (df["prot_effect"] > 0)
) | (
    (df["rna_log2fc"] < 0) & (df["prot_effect"] < 0)
)

print(df["concordant"].value_counts())
display(df[
    ["gene", "rna_log2fc", "prot_effect", "concordant"]
].head())

concordant
False    2276
True     2120
Name: count, dtype: int64


,gene,rna_log2fc,prot_effect,concordant
0,A2M,-1.6,-1.378499,True
1,A2ML1,0.8,-1.477915,False
2,A4GALT,0.2,-0.409790,False
3,AACS,-0.3,0.120361,False
4,AADAC,-2.9,-0.454076,True


### 3.4 — TODO: multi-evidence score
Build a per-layer magnitude for each of the three layers, then score with `multi_evidence_score` and `EQUAL_WEIGHTS`. If you change the weights, justify it in your report.

In [19]:
df["transcriptomic"] = df["rna_log2fc"].abs()
df["proteomic"] = df["prot_effect"].abs()
df["genomic"] = df["mutation_freq"]

df["score"] = multi_evidence_score(
    df,
    ["transcriptomic", "proteomic", "genomic"],
    EQUAL_WEIGHTS
)

ranked_ov = df.sort_values("score", ascending=False)

display(ranked_ov.head(15))

,gene,rna_log2fc,rna_pval,prot_effect,prot_pval,mutation_freq,concordant,transcriptomic,proteomic,genomic,score
3265,RUNX1T1,-4.2,2.746724e-10,-1.871441,3.194047e-04,0.048780,True,4.2,1.871441,0.048780,0.979641
2751,PGR,-4.2,1.798221e-08,-4.117847,9.502484e-03,0.036585,True,4.2,4.117847,0.036585,0.977214
1017,DMD,-4.3,1.389623e-09,-2.278315,1.401506e-08,0.036585,True,4.3,2.278315,0.036585,0.970921
3716,SYNPO2,-4.3,2.051925e-11,-2.273782,1.356978e-07,0.036585,True,4.3,2.273782,0.036585,0.970845
2098,MAOA,-3.4,1.947199e-19,-1.696352,1.023953e-07,0.036585,True,3.4,1.696352,0.036585,0.961101
2954,PROS1,-4.5,1.426385e-09,-1.201194,2.832326e-08,0.036585,True,4.5,1.201194,0.036585,0.952305
3700,SVEP1,-3.6,4.549645e-19,-0.904576,3.576036e-06,0.060976,True,3.6,0.904576,0.060976,0.949803
4027,TTC3,-2.2,3.756645e-10,-1.358566,1.353952e-02,0.048780,True,2.2,1.358566,0.048780,0.945708
795,COL3A1,-3.0,1.751277e-03,-1.175147,1.417956e-02,0.036585,True,3.0,1.175147,0.036585,0.942751
15,ABCA8,-7.8,3.869384e-33,-2.420279,1.852775e-08,0.024390,True,7.8,2.420279,0.024390,0.931870


### 3.5 — TODO: rank, inspect, and export
Sort by score, take the top ~15, and **export the ranked table to `targets_ad.csv`** — this file is the hand-off to Week 3. Check whether known AD genes (APOE, TREM2, BIN1, CLU, PICALM …) are recovered, and look at any `concordant == False` genes in your top hits.

In [20]:
# 3.5 — rank and export top targets

top15 = ranked_ov.head(15).copy()

display(top15)

top15.to_csv("targets_ov.csv", index=False)

print("Saved: targets_ov.csv")

,gene,rna_log2fc,rna_pval,prot_effect,prot_pval,mutation_freq,concordant,transcriptomic,proteomic,genomic,score
3265,RUNX1T1,-4.2,2.746724e-10,-1.871441,3.194047e-04,0.048780,True,4.2,1.871441,0.048780,0.979641
2751,PGR,-4.2,1.798221e-08,-4.117847,9.502484e-03,0.036585,True,4.2,4.117847,0.036585,0.977214
1017,DMD,-4.3,1.389623e-09,-2.278315,1.401506e-08,0.036585,True,4.3,2.278315,0.036585,0.970921
3716,SYNPO2,-4.3,2.051925e-11,-2.273782,1.356978e-07,0.036585,True,4.3,2.273782,0.036585,0.970845
2098,MAOA,-3.4,1.947199e-19,-1.696352,1.023953e-07,0.036585,True,3.4,1.696352,0.036585,0.961101
2954,PROS1,-4.5,1.426385e-09,-1.201194,2.832326e-08,0.036585,True,4.5,1.201194,0.036585,0.952305
3700,SVEP1,-3.6,4.549645e-19,-0.904576,3.576036e-06,0.060976,True,3.6,0.904576,0.060976,0.949803
4027,TTC3,-2.2,3.756645e-10,-1.358566,1.353952e-02,0.048780,True,2.2,1.358566,0.048780,0.945708
795,COL3A1,-3.0,1.751277e-03,-1.175147,1.417956e-02,0.036585,True,3.0,1.175147,0.036585,0.942751
15,ABCA8,-7.8,3.869384e-33,-2.420279,1.852775e-08,0.024390,True,7.8,2.420279,0.024390,0.931870


Saved: targets_ov.csv


### 3.6 — Interpretation (write-up, goes in your report)
Answer these in your 3–4 page report — this is the 40% interpretation payload:

1. **Weighting.** You used equal weights. Argue for keeping them equal *or* for up-weighting a layer (e.g. GWAS as germline/causal-leaning vs. transcriptomics as possibly downstream). There's no single right answer — only reasoned vs. unreasoned.
2. **Top targets.** Which known AD genes did you recover (APOE, TREM2, BIN1, CLU, PICALM …)? Any non-obvious hit worth a second look?
3. **Read a discordant gene.** Pick a `concordant == False` gene in your top hits (strong RNA/GWAS, flat protein). What biology could explain RNA and protein disagreeing?
4. **Limitation.** This was **gene-level, cross-cohort** integration — unmatched — so you *cannot* make per-patient claims. Contrast this with the matched CPTAC case from Part 1.

---
### Submit (Part 3 only)
1. `Kernel → Restart & Run All` — confirm the whole notebook runs top to bottom.
2. Commit **this notebook**, **`targets_ad.csv`**, and a short **README** (your name + anything that didn't work) to your `biot6900` repo.
3. Push, confirm the files appear on github.com, then **post your repo link on Canvas.**

Grading follows the course 60 / 40 split — 60 execution, 40 interpretation & communication. Partial credit for a correct approach even with minor technical errors: if a step wouldn't run, say what you were trying to do and what happened.